[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/eygpcr/biyofizik2026-martini/blob/main/notebooks/06_bentopy.ipynb)

# Oturum 6 — `bentopy` ile Kalabalık Hücresel Sistemler

**Biyofizik 2026 Kursu · Dr. Öğr. Üyesi Ekrem Yaşar**

Bu not defterinde Martini resmî öğretim materyali doğrudan uygulanmaktadır:
[Bentopy Tutorial — cgmartini.nl](https://cgmartini.nl/docs/tutorials/Martini3/Bentopy/)

Kurs boyunca izlenen ölçek gelişimi:

| Oturum | Sistem | Karakteristik uzunluk |
|---|---|---|
| 2 | Çözelti içinde tek lizozim molekülü | yaklaşık 5 nm |
| 3 | Membranda tek reseptör, atomistik | yaklaşık 10 nm |
| 4 | Membranda tek reseptör, kaba-taneli | yaklaşık 12 nm |
| 6 | Kalabalık, çok bölmeli hücresel sistem | 40 nm |

Uygulamada kullanılan model proteinlerden biri lizozimdir; Oturum 2'de
atomistik olarak hazırlanan sistemin kaba-taneli karşılığı burada yüzlerce
kopya hâlinde kullanılmaktadır.


---
## Kalabalık ortam koşullarının önemi

Moleküler simülasyonların büyük bölümü proteinleri seyreltik çözelti
koşullarında incelemektedir. Hücre içi ortam bu varsayımdan belirgin biçimde
ayrılmaktadır:

- Sitoplazmada toplam makromolekül derişimi yaklaşık 300 g/L düzeyindedir;
  hacmin %20–30'u makromoleküller tarafından işgal edilmektedir.
- Biyolojik membranlarda protein/lipit oranı yüksektir.
- Kalabalık koşulları difüzyonu yavaşlatmakta ve bağlanma dengelerini
  kaydırmaktadır.

**Yöntemsel güçlük.** Çok sayıda makromolekülün çakışma oluşturmadan, uygun
yönelimlerle, hedeflenen derişimde ve belirli hücresel bölmelere
yerleştirilmesi elle yapılabilecek bir işlem değildir.

### `bentopy` bileşenleri

| Komut | İşlevi |
|---|---|
| `bentopy-mask` | Var olan bir yapıdan bölme maskeleri üretilmesi |
| `bentopy-pack` | Yapıların bölmelere çakışmasız olarak yerleştirilmesi |
| `bentopy-render` | Yerleşim planından koordinat ve topolojinin üretilmesi |
| `bentopy-merge` | Paketlenen yapıların var olan bir sistemle birleştirilmesi |
| `bentopy-solvate` | Kalan boşluğun çözücü ve iyonlarla doldurulması |

Yerleşim, `.bent` uzantılı bir yapılandırma dosyasıyla tanımlanmaktadır.
Söz dizimi başvurusu:
[Reference for `.bent` files](https://github.com/marrink-lab/bentopy/wiki/Reference-for-bent)


---
## 1. Google Drive bağlanması

Colab çalışma zamanı sonlandığında üretilen dosyalar silinmektedir. Bu nedenle
çıktılar Google Drive üzerinde kalıcı bir klasöre kaydedilecektir.

**Önemli.** Bu oturumdaki üç uygulama **aynı dosya adlarını** kullanmaktadır
(`placements.json`, `system.gro`, `topol.top`, `solvated_system.gro`). Her
uygulama bir öncekinin çıktısının üzerine yazmaktadır. Bu nedenle her uygulama
kendi alt klasörüne kaydedilmektedir.

Oluşturulacak klasör yapısı:

```
Drive'ım/
└── Biyofizik2026_Martini/
    ├── martini_input/          <- Oturum 4
    └── bentopy/                <- bu oturum
        ├── uygulama1_kutu/
        ├── uygulama2_membran/
        ├── uygulama3_bolmeler/
        └── gorseller/
```


In [ ]:
from google.colab import drive
import os, shutil, glob

drive.mount('/content/drive')

# --- Kalici klasor yapisi (Google Drive) ---
DRIVE_KOK = '/content/drive/MyDrive/Biyofizik2026_Martini'
OTURUM    = os.path.join(DRIVE_KOK, 'bentopy')
D_UYG1    = os.path.join(OTURUM, 'uygulama1_kutu')
D_UYG2    = os.path.join(OTURUM, 'uygulama2_membran')
D_UYG3    = os.path.join(OTURUM, 'uygulama3_bolmeler')
D_GORSEL  = os.path.join(OTURUM, 'gorseller')

for d in (DRIVE_KOK, OTURUM, D_UYG1, D_UYG2, D_UYG3, D_GORSEL):
    os.makedirs(d, exist_ok=True)

print('Drive klasoru:', OTURUM)
for d in (D_UYG1, D_UYG2, D_UYG3, D_GORSEL):
    print('  -', os.path.basename(d))


In [ ]:
def kaydet(desenler, hedef, sessiz=False):
    """Verilen dosya desenlerini Drive'daki hedef klasore kopyalar."""
    kopyalanan = []
    for desen in desenler:
        for dosya in glob.glob(desen):
            if os.path.isfile(dosya):
                shutil.copy(dosya, hedef)
                kopyalanan.append(os.path.basename(dosya))
    if not sessiz:
        if kopyalanan:
            print(f'Drive\'a kaydedildi ({os.path.basename(hedef)}/):')
            for k in sorted(kopyalanan):
                print('  -', k)
        else:
            print('Kopyalanacak dosya bulunamadi:', desenler)
    return kopyalanan


---
## 2. Yazılım kurulumu


In [ ]:
%%capture
!pip install -q bentopy
!apt-get -qq update && apt-get -qq install -y gromacs


In [ ]:
!bentopy-pack --help 2>&1 | head -15
print('---')
!which bentopy-pack bentopy-render bentopy-solvate bentopy-mask bentopy-merge


---
## 3. Tutorial dosyalarının indirilmesi

Arşiv, uygulamalarda kullanılacak yapıları, topolojileri ve `.mdp`
dosyalarını içermektedir (yaklaşık 3.6 MB).

Tüm komutlar `tutorial_files/` dizini içinden çalıştırılmalıdır.


In [ ]:
os.chdir('/content')
!wget -q https://cgmartini-library.s3.ca-central-1.amazonaws.com/0_Tutorials/m3_tutorials/Bentopy/tutorial_files.tar.gz
!tar -xzf tutorial_files.tar.gz

os.chdir('/content/tutorial_files')
print('Calisma dizini:', os.getcwd())
!ls structures/ topology/ mdp_files/


### Görselleştirme yardımcı fonksiyonu

Kurulan sistemlerin doğruluğu, z ekseni boyunca bileşen dağılımına bakılarak
denetlenebilir. Bu fonksiyon her uygulamanın sonunda kullanılacaktır.

Uygulama 2 ve 3'te bu çizim, `[ compartments ]` bölümünde tanımlanan konum
kurallarının gerçekten işleyip işlemediğini doğrudan göstermektedir.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

LIPIT  = {'POPC','POPE','POPS','POPG','CHOL','DOPC','DPPC','DOPE'}
COZUCU = {'W','WF','PW'}
IYON   = {'NA','CL','NA+','CL-','ION','K'}

def gro_oku(path, max_atom=1_200_000):
    """GRO dosyasindan rezidu adlarini ve z koordinatlarini okur."""
    satirlar = open(path).read().splitlines()
    n = int(satirlar[1])
    atomlar = satirlar[2:2+n]
    if n > max_atom:
        atomlar = atomlar[::(n // max_atom + 1)]
    res = np.array([s[5:10].strip() for s in atomlar])
    z   = np.array([float(s[36:44]) for s in atomlar])
    return res, z

def profil_ciz(gro, png, baslik):
    res, z = gro_oku(gro)
    diger = list(LIPIT | COZUCU | IYON)
    gruplar = {
        'Protein': ~np.isin(res, diger),
        'Lipit'  : np.isin(res, list(LIPIT)),
        'Cozucu' : np.isin(res, list(COZUCU)),
        'Iyon'   : np.isin(res, list(IYON)),
    }
    kenar = np.linspace(z.min(), z.max(), 120)
    plt.figure(figsize=(9, 4.5))
    for ad, maske in gruplar.items():
        if maske.sum() == 0:
            continue
        plt.hist(z[maske], bins=kenar, histtype='step', lw=1.6,
                 label=f'{ad} (n={maske.sum():,})')
    plt.xlabel('z (nm)'); plt.ylabel('Parcacik sayisi')
    plt.title(baslik); plt.legend(); plt.grid(alpha=.3)
    plt.tight_layout()
    plt.savefig(png, dpi=150)
    plt.show()
    print('Kaydedildi:', png)

def parcacik_sayisi(gro):
    return int(open(gro).read().splitlines()[1])


---
## 4. Uygulama 1 — Kutu içinde protein paketleme

Sitoplazmik yoğunlukta, homojen dağılımlı bir protein sistemi kurulmaktadır:
40 × 40 × 40 nm boyutlarında bir kutuya 650 lizozim molekülü.

### Yapılandırma dosyası

`.bent` dosyası beş bölümden oluşmaktadır:

| Bölüm | İçeriği |
|---|---|
| `[ general ]` | Sistem başlığı ve rastgelelik tohumu |
| `[ space ]` | Kutu boyutları ve paketleme ızgara çözünürlüğü |
| `[ includes ]` | Topolojiye eklenecek kuvvet alanı dosyaları |
| `[ compartments ]` | Yerleştirmenin yapılacağı hacimlerin tanımı |
| `[ segments ]` | Hangi yapıdan kaç kopyanın hangi bölmeye konacağı |


In [ ]:
bent = '''[ general ]
title "Proteins in a box"
seed 0

[ space ]
dimensions 40, 40, 40
resolution 0.5

[ includes ]
"topology/martini_v3.0.0.itp"
"topology/martini_v3.0.0_ions_v1.itp"
"topology/martini_v3.0.0_solvents_v1.itp"
"topology/lysozyme.itp"

[ compartments ]
system is all

[ segments ]
LYZ 650 from "structures/lysozyme.pdb" in system
'''
open('simple_packing.bent','w').write(bent)
print(bent)


In [ ]:
!bentopy-pack simple_packing.bent placements.json


In [ ]:
!bentopy-render placements.json system.gro -t topol.top

if os.path.exists('system.gro'):
    print(f'Paketlenmis sistem: {parcacik_sayisi("system.gro"):,} parcacik')


In [ ]:
!bentopy-solvate -i system.gro -o solvated_system.gro \
    -s NA:0.15M -s CL:0.15M \
    --charge neutral \
    -t topol.top


In [ ]:
if os.path.exists('solvated_system.gro'):
    print(f'Solvatlanmis sistem: {parcacik_sayisi("solvated_system.gro"):,} parcacik')
    print()
    print('--- topol.top ---')
    print(open('topol.top').read())


### Görselleştirme ve kaydetme

Beklenen görünüm: protein z ekseni boyunca **düzgün dağılmış** olmalıdır.
Uygulama 1'in amacı homojen paketlemedir; bir bölge zenginleşmesi
görülmemelidir.


In [ ]:
profil_ciz('solvated_system.gro',
           os.path.join(D_GORSEL, 'uygulama1_z_profili.png'),
           'Uygulama 1 - kutuda homojen protein paketlemesi')


In [ ]:
kaydet(['simple_packing.bent','placements.json','system.gro',
        'topol.top','solvated_system.gro'], D_UYG1)


**Tartışma sorusu.** Oturum 2'de tek bir lizozim molekülü için kurulan
atomistik sistemin atom sayısı ile bu sistemin parçacık sayısı
karşılaştırıldığında hangi ölçek farkı ortaya çıkmaktadır?


---
## 5. Uygulama 2 — Membran çevresinde konuma bağlı paketleme

Bu uygulamada proteinler yalnızca sayıca değil, konumsal kurala göre de
yerleştirilmektedir: lizozim çözücü hacmine dağıtılırken, ubikitin yalnızca
membran yüzeyine yakın bölgede konumlandırılmaktadır.

### Maskenin üretilmesi

Var olan bir membran yapısından bölme maskesi çıkarılmaktadır. İlk komut
etiketlemenin görsel olarak denetlenmesini sağlamakta, ikinci komut
kullanılacak maske dosyasını üretmektedir.


In [ ]:
!bentopy-mask structures/membrane.gro --visualize-labels labels.gro
!bentopy-mask structures/membrane.gro -l 1:membrane_mask.npz
!ls -lh membrane_mask.npz labels.gro


### Yapılandırma dosyası

**Kavramsal not.** `[ compartments ]` bölümündeki tanımlar:

- `membrane from "membrane_mask.npz"` — maske dosyasından bölme tanımlama
- `solvent combines not membrane` — membran dışında kalan hacim
- `close-to-membrane around 5 of membrane` — membran yüzeyinden 5 nm
  mesafedeki kabuk

Bu yaklaşım, periferik membran proteinlerinin fizyolojik dağılımının
modellenmesine olanak vermektedir.


In [ ]:
bent = '''[ general ]
title "Proteins around a membrane"
seed 0

[ space ]
dimensions 40, 40, 40
resolution 0.5

[ includes ]
"topology/martini_v3.0.0.itp"
"topology/martini_v3.0.0_ions_v1.itp"
"topology/martini_v3.0.0_solvents_v1.itp"
"topology/martini_v3.0.0_phospholipids_v1.itp"
"topology/lysozyme.itp"
"topology/ubiquitin.itp"

[ compartments ]
membrane from "membrane_mask.npz"
solvent combines not membrane
close-to-membrane around 5 of membrane

[ segments ]
LYZ:lyz 300 from "structures/lysozyme.pdb" in solvent
UBQ:ubq 100 from "structures/ubiquitin.pdb" in close-to-membrane
'''
open('membrane_packing.bent','w').write(bent)
print(bent)


In [ ]:
!bentopy-pack membrane_packing.bent placements.json
!bentopy-render placements.json packed_proteins.gro -t topol.top


### Membranla birleştirme

**Dikkat.** `bentopy-merge` işleminden sonra lipit sayısının topoloji
dosyasına elle eklenmesi gerekmektedir; birleştirilen membran yapısı
`bentopy` tarafından üretilmediğinden topolojide otomatik olarak yer
almamaktadır.


In [ ]:
!bentopy-merge packed_proteins.gro structures/membrane.gro -o system.gro
!echo "POPC    5408" >> topol.top
!tail -8 topol.top


In [ ]:
!bentopy-solvate -i system.gro -o solvated_system.gro -t topol.top \
    -s NA:0.15M -s CL:0.15M --charge neutral

if os.path.exists('solvated_system.gro'):
    print(f'\nSolvatlanmis sistem: {parcacik_sayisi("solvated_system.gro"):,} parcacik')


### Görselleştirme ve kaydetme

Beklenen görünüm: lipitler kutunun ortasında dar bir bant (çift tabaka)
oluşturmalı, protein dağılımı ise membran çevresinde belirgin biçimde
**zenginleşmelidir**. Bu, `close-to-membrane` bölme tanımının işlediğinin
doğrudan kanıtıdır.


In [ ]:
profil_ciz('solvated_system.gro',
           os.path.join(D_GORSEL, 'uygulama2_z_profili.png'),
           'Uygulama 2 - membran cevresinde konuma bagli paketleme')


In [ ]:
kaydet(['membrane_packing.bent','placements.json','membrane_mask.npz','labels.gro',
        'packed_proteins.gro','system.gro','topol.top','solvated_system.gro'], D_UYG2)


---
## 6. Uygulama 3 — Çok bölmeli sistem

Çift membranla ayrılmış iki bölme tanımlanmakta ve her bölmeye farklı protein
yerleştirilmektedir. Süre elverdiği takdirde yürütülecektir.

Maskelerin üretilmesinde `-b` seçeneği bölme etiketlerinin
görselleştirilmesini, `-l` seçenekleri ise her etiket için ayrı maske dosyası
üretilmesini sağlamaktadır.


In [ ]:
!bentopy-mask structures/double_membrane.gro -b compartment_labels.gro
!bentopy-mask structures/double_membrane.gro \
    -l  -1:A_mask.npz \
    -l  -2:B_mask.npz \
    -l 1,2:membrane_mask.npz
!ls -lh A_mask.npz B_mask.npz membrane_mask.npz


In [ ]:
bent = '''[ general ]
title "Proteins in different compartments"
seed 0

[ space ]
dimensions 40, 40, 40
resolution 0.5

[ includes ]
"topology/martini_v3.0.0.itp"
"topology/martini_v3.0.0_ions_v1.itp"
"topology/martini_v3.0.0_solvents_v1.itp"
"topology/martini_v3.0.0_phospholipids_v1.itp"
"topology/lysozyme.itp"
"topology/ubiquitin.itp"

[ compartments ]
membrane from "membrane_mask.npz"
A from "A_mask.npz"
B from "B_mask.npz"
membrane-neighborhood around 4 of membrane
B-close-to-membrane combines membrane-neighborhood and B

[ segments ]
LYZ:lyz 200 from "structures/lysozyme.pdb" in A
UBQ:ubq 100 from "structures/ubiquitin.pdb" in B-close-to-membrane
'''
open('compartment_packing.bent','w').write(bent)
print(bent)


In [ ]:
!bentopy-pack compartment_packing.bent placements.json
!bentopy-render placements.json packed_proteins.gro -t topol.top
!bentopy-merge packed_proteins.gro structures/double_membrane.gro -o system.gro
!echo "POPC    10816" >> topol.top
!bentopy-solvate -i system.gro -o solvated_system.gro -t topol.top \
    -s NA:0.15M -s CL:0.15M --charge neutral


### Görselleştirme ve kaydetme

Beklenen görünüm: **iki** lipit bandı (çift membran) ve bunlar arasında
bölmeye özgü protein dağılımı. Lizozim yalnızca A bölmesinde, ubikitin ise
B bölmesinin membrana yakın kesiminde bulunmalıdır.


In [ ]:
if os.path.exists('solvated_system.gro'):
    print(f'Cok bolmeli sistem: {parcacik_sayisi("solvated_system.gro"):,} parcacik')

profil_ciz('solvated_system.gro',
           os.path.join(D_GORSEL, 'uygulama3_z_profili.png'),
           'Uygulama 3 - cift membranli cok bolmeli sistem')


In [ ]:
kaydet(['compartment_packing.bent','placements.json','compartment_labels.gro',
        'A_mask.npz','B_mask.npz','membrane_mask.npz',
        'packed_proteins.gro','system.gro','topol.top','solvated_system.gro'], D_UYG3)


---
## 7. Kurulan sistemin simülasyona hazırlanması

Kursta üretim simülasyonu koşulmamaktadır. Aşağıdaki adımlar, kurulan
sistemin doğrudan kullanılabilir olduğunu göstermek amacıyla verilmiştir;
hücreler yorum satırı hâlinde bırakılmıştır.

Tam betik: [`06_bentopy/kodlar/simulasyon.sh`](https://github.com/eygpcr/biyofizik2026-martini/blob/main/06_bentopy/kodlar/simulasyon.sh)


In [ ]:
# Enerji minimizasyonu
# !gmx grompp -f mdp_files/em.mdp -c solvated_system.gro -p topol.top -o em.tpr
# !gmx mdrun -v -deffnm em

# Indeks gruplarinin tanimlanmasi, dengeleme ve uretim asamalari icin
# kodlar/simulasyon.sh dosyasina bakiniz.


---
## 8. Drive klasörünün özeti

Bu oturumda üretilen tüm dosyalar Google Drive'a kaydedilmiştir.


In [ ]:
print('Google Drive icerigi:', OTURUM.replace('/content/drive/MyDrive', "Drive'im"))
print()
toplam = 0
for alt in ('uygulama1_kutu', 'uygulama2_membran', 'uygulama3_bolmeler', 'gorseller'):
    yol = os.path.join(OTURUM, alt)
    dosyalar = sorted(os.listdir(yol))
    print(f'{alt}/ ({len(dosyalar)} dosya)')
    for d in dosyalar:
        boyut = os.path.getsize(os.path.join(yol, d))
        toplam += boyut
        print(f'    {d:<30} {boyut:>12,} bayt')
    print()
print(f'Toplam: {toplam/1e6:.1f} MB')


---
## Sorun giderme

| Sorun | Çözümü |
|---|---|
| `pip install bentopy` derleme hatası veriyor | Önceden derlenmiş paket bulunamamıştır; [rustup](https://rustup.rs/) ile Rust derleyicisi kurulmalıdır |
| `bentopy-pack: command not found` | Kurulum hücresi yeniden çalıştırılmalıdır |
| Maske dosyası üretilmiyor | Çalışma dizininin `/content/tutorial_files` olduğu doğrulanmalıdır |
| Paketleme çok uzun sürüyor | `[ segments ]` bölümündeki kopya sayısı azaltılabilir |
| `MessageError: credential propagation was unsuccessful` | Drive bağlama izni verilmedi; 1. bölüm yeniden çalıştırılmalıdır |
| Drive'da yer kalmadı | Solvatlanmış sistemler büyüktür; eski oturum klasörleri silinebilir |

Oturum için ayrılan süre sınırlıdır; sorun yaşanması hâlinde uygulama
sonlandırılarak aşağıdaki materyaller kullanılacaktır:

- Komutlar ve `.bent` yapılandırma dosyaları: [`06_bentopy/kodlar/`](https://github.com/eygpcr/biyofizik2026-martini/tree/main/06_bentopy/kodlar)
- Uygulamanın tam kaydı: [`VIDEO.md`](https://github.com/eygpcr/biyofizik2026-martini/blob/main/06_bentopy/VIDEO.md)

---

## Kaynaklar

- [Bentopy Tutorial — cgmartini.nl](https://cgmartini.nl/docs/tutorials/Martini3/Bentopy/)
- [bentopy deposu](https://github.com/marrink-lab/bentopy) ve [wiki](https://github.com/marrink-lab/bentopy/wiki)
- [`.bent` dosya biçimi başvurusu](https://github.com/marrink-lab/bentopy/wiki/Reference-for-bent)
- [`bentopy-solvate` belgelendirmesi](https://github.com/marrink-lab/bentopy/blob/main/src/solvate/README.md)

Ayrıca bkz. [`ILERI_OKUMA.md`](https://github.com/eygpcr/biyofizik2026-martini/blob/main/ILERI_OKUMA.md)
